# Вариант C. CDI → ЕГРН по адресу и площади

Это оптимизированная версия.

Порядок работы:

1. `full_address` передаётся в CDI без изменений;
2. CDI возвращает ФИАС дома и части адреса;
3. по ФИАС дома из ЕГРН быстро выбирается небольшой список кандидатов;
4. кандидаты проверяются по частям адреса и площади;
5. ЕГРН присоединяется, только если остался один кадастровый объект.

ФИАС используется только для быстрого поиска кандидатов. Окончательное решение принимается по адресу и площади.


In [ ]:
%pip install pandas sqlalchemy "psycopg[binary]" oracledb

In [ ]:
import json
import re
from pathlib import Path
import oracledb
import pandas as pd
from sqlalchemy import URL, create_engine, text

pd.set_option('display.max_columns', 100)

In [ ]:
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / 'notebooks' / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR / 'notebooks'
else:
    NOTEBOOK_DIR = CURRENT_DIR

PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == 'notebooks'
    else NOTEBOOK_DIR
)
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Корень проекта:', PROJECT_ROOT)
print('Папка результатов:', OUTPUT_DIR)

# 1. Подключение к Сфере


In [ ]:
CREDENTIALS_PATH = NOTEBOOK_DIR / 'уч данные.txt'

def read_credentials(path):
    if not path.exists():
        raise FileNotFoundError(f'Не найден файл с учётными данными: {path}')

    credentials = {}
    for line_number, raw_line in enumerate(
        path.read_text(encoding='utf-8-sig').splitlines(),
        start=1,
    ):
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' not in line:
            raise ValueError(
                f'Строка {line_number}: ожидается запись КЛЮЧ=значение'
            )

        key, value = line.split('=', 1)
        credentials[key.strip()] = value.strip()

    return credentials

credentials = read_credentials(CREDENTIALS_PATH)

sphere_required = [
    'SPHERE_HOST',
    'SPHERE_DATABASE',
    'SPHERE_USER',
    'SPHERE_PASSWORD',
]
sphere_missing = [key for key in sphere_required if not credentials.get(key)]
if sphere_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(sphere_missing)
    )

SPHERE_HOST = credentials['SPHERE_HOST']
SPHERE_PORT = int(credentials.get('SPHERE_PORT', '5432'))
SPHERE_DATABASE = credentials['SPHERE_DATABASE']
SPHERE_USER = credentials['SPHERE_USER']
SPHERE_PASSWORD = credentials['SPHERE_PASSWORD']

connection_url = URL.create(
    drivername='postgresql+psycopg',
    username=SPHERE_USER,
    password=SPHERE_PASSWORD,
    host=SPHERE_HOST,
    port=SPHERE_PORT,
    database=SPHERE_DATABASE,
)
engine = create_engine(connection_url, pool_pre_ping=True)

print('Учётные данные прочитаны, подключение к Сфере создано')


In [ ]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

# 2. Подключение к Oracle КХД



In [ ]:
khd_required = [
    'KHD_HOST',
    'KHD_SERVICE_NAME',
    'KHD_USER',
    'KHD_PASSWORD',
]
khd_missing = [key for key in khd_required if not credentials.get(key)]
if khd_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(khd_missing)
    )

KHD_HOST = credentials['KHD_HOST']
KHD_PORT = int(credentials.get('KHD_PORT', '1521'))
KHD_SERVICE_NAME = credentials['KHD_SERVICE_NAME']
KHD_USER = credentials['KHD_USER']
KHD_PASSWORD = credentials['KHD_PASSWORD']
KHD_DATA_SCHEMA = credentials.get('KHD_DATA_SCHEMA', 'DM_RISK_AVATAR')

khd_dsn = oracledb.makedsn(
    KHD_HOST,
    KHD_PORT,
    service_name=KHD_SERVICE_NAME,
)
khd_connection = oracledb.connect(
    user=KHD_USER,
    password=KHD_PASSWORD,
    dsn=khd_dsn,
)

print('Подключение к КХД создано')


In [ ]:
# проверяем доступ к ЕГРН
khd_schema_for_check = KHD_DATA_SCHEMA.upper()
if not khd_schema_for_check.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

with khd_connection.cursor() as cursor:
    cursor.execute(
        f'select 1 from {khd_schema_for_check}.EGRN_DATA '
        'where rownum = 1'
    )
    cursor.fetchone()

print('Таблица EGRN_DATA доступна')


# 3. SQL Сфера, расширенный подход


In [ ]:
expanded_sql = r"""
/*
запускать в сфере

запрос собирает все неудаленные объекты недвижимости
если объект связан с подходящим договором данные договора заполняются
если связь не найдена объект остается в результате с пустыми полями договора

одна строка для связанного объекта означает объект в одном договоре
одна строка для несвязанного объекта означает его последнюю версию характеристик
*/

with task_candidates as (
    /* отбираем подходящие задачи оформления */
    select
        t.id as task_id,
        r.id as request_id,
        c.id as contract_id,
        row_number() over (
            partition by c.id
            order by
                coalesce(
                    t.d_conclusion_ins_contract::timestamp with time zone,
                    t.d_create,
                    t.d_change
                ) desc nulls last,
                t.d_create desc nulls last,
                t.d_change desc nulls last,
                t.id desc
        ) as task_number
    from bps_request_ins_task t
    join bps_request_ins r
        on r.id = t.request_ins_id
    join bps_contract c
        on c.id = r.contract_id
    where t.task_type = 'draft_contract'
      and t.status = 'operational_archive'
      and (
          t.ins_document_type = 'new_ins_contract'
          or t.ins_document_type = 'ins_contract_prolong'
          or t.ins_document_type is null
      )
      and t.ins_refuse is not true
      and t.d_delete is null
      and r.d_delete is null
      and c.d_delete is null
),

selected_tasks as (
    /* оставляем последнюю подходящую задачу каждого договора */
    select
        task_id,
        request_id,
        contract_id
    from task_candidates
    where task_number = 1
),

linked_object_candidates as (
    /* находим недвижимость в выбранных задачах */
    select
        ch.insurance_object_id as object_id,
        ch.id as characteristics_id,
        link.id as task_object_link_id,
        selected.task_id,
        selected.request_id,
        selected.contract_id,
        row_number() over (
            partition by selected.task_id, ch.insurance_object_id
            order by
                link.d_change desc nulls last,
                link.d_create desc nulls last,
                ch.version_start_date desc nulls last,
                ch.version_number desc nulls last,
                link.id desc
        ) as link_number
    from selected_tasks selected
    join bps_request_ins_task_insurance_object link
        on link.parent_id = selected.task_id
    join base_insurance_object_characteristics ch
        on ch.id = link.characteristics_id
    join base_insurance_object obj
        on obj.id = ch.insurance_object_id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

selected_links as (
    /* убираем повторные связи одного объекта с одной задачей */
    select
        object_id,
        characteristics_id,
        task_object_link_id,
        task_id,
        request_id,
        contract_id
    from linked_object_candidates
    where link_number = 1
),

object_versions as (
    /* нумеруем версии характеристик каждого объекта */
    select
        obj.id as object_id,
        ch.id as characteristics_id,
        row_number() over (
            partition by obj.id
            order by
                ch.version_is_active desc nulls last,
                ch.version_number desc nulls last,
                ch.version_start_date desc nulls last,
                ch.id desc nulls last
        ) as version_number
    from base_insurance_object obj
    left join base_insurance_object_characteristics ch
        on ch.insurance_object_id = obj.id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

dataset_keys as (
    /* сохраняем все найденные связи с договорами */
    select
        linked.object_id,
        linked.characteristics_id,
        linked.task_object_link_id,
        linked.task_id,
        linked.request_id,
        linked.contract_id,
        'linked'::text as row_source
    from selected_links linked

    union all

    /* добавляем объекты для которых подходящий договор не найден */
    select
        version.object_id,
        version.characteristics_id,
        null::integer as task_object_link_id,
        null::integer as task_id,
        null::integer as request_id,
        null::integer as contract_id,
        'not_linked'::text as row_source
    from object_versions version
    where version.version_number = 1
      and not exists (
          select 1
          from selected_links linked
          where linked.object_id = version.object_id
      )
),

object_link_profile as (
    /* считаем со сколькими договорами связан объект */
    select
        object_id,
        count(distinct contract_id) as contract_count
    from selected_links
    group by object_id
),

selected_characteristics as (
    /* ограничиваем расчет условий версиями из итоговой выборки */
    select distinct characteristics_id
    from dataset_keys
    where characteristics_id is not null
),

condition_summary as (
    /* сворачиваем варианты условий в одну строку */
    select
        cond.characteristics_id,
        count(*) as condition_count,
        count(cond.insured_sum) as filled_insured_sum_count,
        count(distinct cond.insured_sum) filter (
            where cond.insured_sum is not null
        ) as distinct_insured_sum_count,
        min(cond.insured_sum) as minimum_insured_sum,
        max(cond.insured_sum) as maximum_insured_sum,
        count(distinct cond.insured_sum_currency) filter (
            where cond.insured_sum_currency is not null
        ) as currency_count,
        string_agg(
            distinct cond.insured_sum_currency,
            ', '
            order by cond.insured_sum_currency
        ) filter (
            where cond.insured_sum_currency is not null
        ) as insured_sum_currency,
        array_agg(
            distinct cond.terms_option_number
            order by cond.terms_option_number
        ) filter (
            where cond.terms_option_number is not null
        ) as terms_option_numbers,
        min(cond.per_occurance_limit) as minimum_per_occurrence_limit,
        max(cond.per_occurance_limit) as maximum_per_occurrence_limit,
        case
            when count(distinct cond.insured_sum) filter (
                where cond.insured_sum is not null
            ) = 1
             and count(distinct cond.insured_sum_currency) filter (
                where cond.insured_sum_currency is not null
            ) <= 1
            then max(cond.insured_sum)
        end as insured_sum
    from base_insurance_object_conditions cond
    join selected_characteristics selected
        on selected.characteristics_id = cond.characteristics_id
    group by cond.characteristics_id
),

raw_result as (
/* собираем исходные поля расширенного датасета */
select
    /* качество строки */
    keys.row_source,
    case
        when coalesce(profile.contract_count, 0) = 0 then 'not_linked'
        when profile.contract_count = 1 then 'linked'
        else 'multiple_contracts'
    end as contract_link_status,
    coalesce(profile.contract_count, 0) as contract_count,
    (keys.contract_id is not null) as has_contract,
    (address.id is not null) as has_address,
    (conditions.insured_sum is not null) as has_target,
    case
        when conditions.condition_count is null then 'no_conditions'
        when conditions.filled_insured_sum_count = 0 then 'target_is_empty'
        when conditions.distinct_insured_sum_count > 1 then 'several_target_values'
        when conditions.currency_count > 1 then 'several_currencies'
        when conditions.insured_sum <= 0 then 'target_is_not_positive'
        else 'target_is_usable'
    end as target_status,

    /* идентификаторы */
    keys.object_id,
    keys.characteristics_id,
    keys.task_object_link_id,
    keys.task_id,
    keys.request_id,
    keys.contract_id,
    obj.geo_address_id,
    contract.contractor_id as policyholder_id,
    request.corporate_crm_id,

    /* целевая страховая сумма */
    conditions.insured_sum,
    conditions.insured_sum_currency,
    conditions.condition_count,
    conditions.filled_insured_sum_count,
    conditions.distinct_insured_sum_count,
    conditions.minimum_insured_sum as condition_min_insured_sum,
    conditions.maximum_insured_sum as condition_max_insured_sum,
    conditions.currency_count as condition_currency_count,
    conditions.terms_option_numbers,

    /* контрольные суммы */
    task_link.insured_sum as task_object_insured_sum,
    task_link.insured_sum_currency as task_object_insured_sum_currency,
    task.total_ins_contract_amount as contract_insured_sum,
    task.curr_ins_contract_amount as contract_amount_currency,
    task.total_ins_contract_premium as contract_premium,
    ch.insurance_value,
    ch.insurance_value_currency,
    ch.insurance_value_basis,
    ch.pledged_value,
    conditions.minimum_per_occurrence_limit,
    conditions.maximum_per_occurrence_limit,

    /* объект */
    obj.obj_type as object_type,
    obj.elementary_obj_type,
    obj.obj_name as object_name,
    obj.description as object_description,
    obj.original_address,
    obj.active as object_is_active,
    obj.d_create as object_create_date,
    obj.d_change as object_change_date,

    /* характеристики объекта */
    ch.version_number as characteristics_version_number,
    ch.version_start_date as characteristics_version_start_date,
    ch.version_end_date as characteristics_version_end_date,
    ch.version_is_active as characteristics_version_is_active,
    ch.ownership_type,
    ch.is_pledged,
    ch.is_leased,
    ch.insured_components,
    ch.activity_types,
    ch.risk_natures,
    ch.insurance_territory,
    ch.has_losses,
    ch.insurance_object_loss_history,
    ch.characteristics ->> 'total_area_sq_m' as total_area,
    ch.characteristics ->> 'occupied_area_sq_m' as occupied_area,
    ch.characteristics ->> 'construction_year' as construction_year,
    ch.characteristics ->> 'last_capital_repair_year' as capital_repair_year,
    ch.characteristics ->> 'total_floors_count' as floors_count,
    ch.characteristics ->> 'occupied_floor' as occupied_floor,
    ch.characteristics ->> 'load_bearing_walls_material' as walls_material,
    ch.characteristics ->> 'interfloor_overlap_material' as overlap_material,
    ch.characteristics ->> 'roofing_material' as roofing_material,
    ch.characteristics ->> 'fire_alarm_system_availability'
        as fire_alarm_system_availability,
    ch.characteristics ->> 'fire_suppression_system_availability'
        as fire_suppression_system_availability,
    ch.characteristics ->> 'nearest_fire_station_distance_km'
        as nearest_fire_station_distance_km,
    ch.characteristics as object_characteristics_json,

    /* адрес */
    address.full_address,
    address.postal_code,
    address.region_id as address_region_id,
    address.area as district,
    address.settlement_type,
    address.settlement,
    address.street_type,
    address.street,
    address.house,
    address.building,
    address.block,
    address.flat,
    address.office,
    address.fias_code,
    address.longitude,
    address.latitude,
    address.address_dgis_id,

    /* договор */
    contract.n_contract as contract_number,
    contract.document_status as contract_status,
    contract.system_type as contract_source_system,
    contract.ins_product_sbs as insurance_product,
    contract.d_sign_contract as contract_sign_date,
    contract.d_start_contract as contract_start_date,
    contract.d_end_contract as contract_end_date,
    contract.prevcontract_id as previous_contract_id,
    contract.rootcontract_id as root_contract_id,
    previous_contract.n_contract as previous_contract_number,
    previous_contract.d_start_contract as previous_contract_start_date,
    previous_contract.d_end_contract as previous_contract_end_date,

    /* задача и заявка */
    task.task_type,
    task.status as task_status,
    task.ins_document_type,
    task.ins_refuse,
    task.d_create as task_create_date,
    task.d_conclusion_ins_contract as contract_conclusion_date,
    task.ins_product as task_product,
    task.industry as task_industry,
    task.subindustry as task_subindustry,
    task.locations_count,
    task.multi_location,
    task.object_description as task_object_description,
    request.business_segment,
    request.sale_channel,
    request.ins_product as request_product,

    /* страхователь и crm */
    policyholder.inn as policyholder_inn,
    policyholder.company_name_short as policyholder_name,
    policyholder.cdi_id as policyholder_cdi_id,
    crm.segment as crm_segment,
    crm.macroindustry as crm_macroindustry,
    crm.industry as crm_industry,
    crm.okved as crm_okved,

    /* дата состояния строки */
    coalesce(
        task.d_conclusion_ins_contract::timestamp with time zone,
        contract.d_sign_contract,
        ch.version_start_date,
        obj.d_create
    ) as as_of_date
from dataset_keys keys
join base_insurance_object obj
    on obj.id = keys.object_id
left join base_insurance_object_characteristics ch
    on ch.id = keys.characteristics_id
left join condition_summary conditions
    on conditions.characteristics_id = keys.characteristics_id
left join bps_request_ins_task_insurance_object task_link
    on task_link.id = keys.task_object_link_id
left join bps_request_ins_task task
    on task.id = keys.task_id
left join bps_request_ins request
    on request.id = keys.request_id
left join bps_contract contract
    on contract.id = keys.contract_id
left join bps_contract previous_contract
    on previous_contract.id = contract.prevcontract_id
left join bps_contractor policyholder
    on policyholder.id = contract.contractor_id
left join bps_corporate_crm crm
    on crm.id = request.corporate_crm_id
left join base_geo_address address
    on address.id = obj.geo_address_id
left join object_link_profile profile
    on profile.object_id = keys.object_id
),

standardized_result as (
    /* приводим результат к общей структуре двух датасетов */
    select
        case
            when raw.contract_id is null then 'not_linked'
            else 'linked'
        end as row_source,
        case
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 0 then 'not_linked'
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 1 then 'linked'
            else 'multiple_contracts'
        end as contract_link_status,
        count(raw.contract_id) over (
            partition by raw.object_id
        ) as contract_count,
        (raw.contract_id is not null) as has_contract,
        (
            raw.geo_address_id is not null
            or nullif(btrim(raw.full_address), '') is not null
            or nullif(btrim(raw.original_address), '') is not null
        ) as has_address,
        (raw.insured_sum is not null) as has_target,
        case
            when raw.condition_count is null
              or raw.condition_count = 0
                then 'no_conditions'
            when raw.condition_min_insured_sum is distinct from
                 raw.condition_max_insured_sum
                then 'several_target_values'
            when coalesce(raw.condition_currency_count, 0) > 1
                then 'several_currencies'
            when raw.insured_sum <= 0
                then 'target_is_not_positive'
            when raw.insured_sum is null
                then 'target_is_empty'
            else 'target_is_usable'
        end as target_status,

        raw.contract_id,
        raw.contract_number,
        raw.previous_contract_id,
        raw.root_contract_id,
        raw.request_id,
        raw.task_id,
        raw.task_object_link_id,
        raw.characteristics_id,
        raw.object_id,
        raw.geo_address_id,
        raw.policyholder_id,
        raw.corporate_crm_id,

        raw.as_of_date,
        raw.contract_conclusion_date,
        raw.contract_sign_date,
        raw.contract_start_date,
        raw.contract_end_date,
        raw.contract_status,
        raw.ins_document_type,
        raw.insurance_product,

        case
            when raw.contract_id is not null then
                count(raw.object_id) over (
                    partition by raw.contract_id
                )
        end as real_estate_objects_in_contract,
        raw.object_name,
        raw.object_description,
        raw.object_type,
        raw.elementary_obj_type,
        raw.total_area,
        raw.occupied_area,
        raw.construction_year,
        raw.capital_repair_year,
        raw.floors_count,
        raw.occupied_floor,
        raw.walls_material,
        raw.overlap_material,
        raw.roofing_material,
        raw.ownership_type,
        raw.is_leased,
        raw.insured_components,
        raw.activity_types,
        raw.risk_natures,
        raw.insurance_territory,

        raw.full_address,
        raw.original_address,
        raw.postal_code,
        raw.address_region_id,
        raw.district,
        raw.settlement,
        raw.street,
        raw.house,
        raw.building,
        raw.block,
        raw.flat,
        raw.office,
        raw.fias_code,
        raw.longitude,
        raw.latitude,
        raw.address_dgis_id,

        raw.policyholder_inn,
        raw.policyholder_name,
        raw.policyholder_cdi_id,
        raw.crm_segment,
        raw.crm_macroindustry,
        raw.crm_industry,
        raw.crm_okved,
        raw.business_segment,
        raw.task_industry,
        raw.task_subindustry,

        raw.contract_insured_sum,
        raw.contract_amount_currency,
        raw.task_object_insured_sum,
        raw.task_object_insured_sum_currency,
        raw.condition_min_insured_sum,
        raw.condition_max_insured_sum,
        raw.insured_sum,
        raw.insured_sum_currency,
        raw.condition_currency_count,
        raw.contract_premium,
        raw.insurance_value,
        raw.insurance_value_currency,
        raw.insurance_value_basis,
        raw.is_pledged,
        raw.pledged_value,
        raw.minimum_per_occurrence_limit,
        raw.maximum_per_occurrence_limit,

        raw.previous_contract_number,
        raw.previous_contract_start_date,
        raw.previous_contract_end_date,

        raw.condition_count,
        raw.characteristics_version_number,
        raw.characteristics_version_start_date,
        raw.characteristics_version_end_date,
        raw.characteristics_version_is_active,
        raw.object_characteristics_json,
        raw.task_type,
        raw.task_status,
        raw.ins_refuse
    from raw_result raw
)

select *
from standardized_result
order by
    has_contract desc,
    as_of_date desc nulls last,
    object_id;

"""


In [ ]:
with engine.connect() as connection:
    expanded_df = pd.read_sql_query(text(expanded_sql), connection)

print('Строк:', len(expanded_df))
print('Колонок:', len(expanded_df.columns))
display(expanded_df.head(3))


# 4. Проверка заполненности

In [ ]:
required_columns = {
    'object_id', 'characteristics_id', 'elementary_obj_type',
    'insured_sum', 'full_address', 'total_area', 'row_source'
}
missing_columns = sorted(required_columns - set(expanded_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных объектов',
        'Строк с договором',
        'Строк без договора',
        'Строк с target',
        'Строк с full_address',
    ],
    'Значение': [
        len(expanded_df),
        expanded_df['object_id'].nunique(dropna=True),
        expanded_df['has_contract'].fillna(False).sum(),
        (~expanded_df['has_contract'].fillna(False)).sum(),
        expanded_df['insured_sum'].notna().sum(),
        expanded_df['full_address'].fillna('').str.strip().ne('').sum(),
    ],
})
display(profile)


In [ ]:
display(expanded_df['row_source'].fillna('empty').value_counts(dropna=False))
display(expanded_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))


# 5. Разбор `full_address` через CDI

CDI здесь используется только как разборщик адреса. ФИАС может находиться в ответе CDI, но в соединении с ЕГРН не участвует.


In [ ]:
# готовим уникальные адреса
sphere_with_row_id = expanded_df.copy()
sphere_with_row_id.insert(0, 'sphere_row_id', range(1, len(sphere_with_row_id) + 1))
sphere_with_row_id['source_address'] = sphere_with_row_id['full_address'].astype('string')
sphere_with_row_id.loc[
    sphere_with_row_id['source_address'].str.strip().eq(''),
    'source_address',
] = pd.NA

unique_addresses = (
    sphere_with_row_id[['source_address']]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
unique_addresses.insert(0, 'address_lookup_id', range(1, len(unique_addresses) + 1))

print('Строк в датасете:', len(sphere_with_row_id))
print('Строк с full_address:', sphere_with_row_id['source_address'].notna().sum())
print('Уникальных full_address:', len(unique_addresses))


In [ ]:
# вызываем рабочую функцию CDI
CDI_SCHEMA = credentials.get('CDI_SCHEMA', 'DM_MOTOR').upper()
CDI_TEXT_FUNCTION = credentials.get('CDI_TEXT_FUNCTION', 'F_GET_CDI_ADDR_BY_TEXT').upper()

for value, label in [(CDI_SCHEMA, 'CDI_SCHEMA'), (CDI_TEXT_FUNCTION, 'CDI_TEXT_FUNCTION')]:
    if not value.replace('_', '').isalnum():
        raise ValueError(f'Некорректное значение {label}')

cdi_function_sql = f'select d.* from {CDI_SCHEMA}.{CDI_TEXT_FUNCTION}(:address_text) d'

if not unique_addresses.empty:
    test_address = unique_addresses.iloc[0]['source_address']
    try:
        with khd_connection.cursor() as cursor:
            cursor.execute(cdi_function_sql, address_text=test_address)
            cursor.fetchmany(1)
    except oracledb.Error as error:
        raise RuntimeError(
            f'Не удалось вызвать {CDI_SCHEMA}.{CDI_TEXT_FUNCTION}. '
            f'Подключение: user={KHD_USER}, service={KHD_SERVICE_NAME}. '
            f'Ошибка Oracle: {error}'
        ) from error

print('Вызов CDI:', cdi_function_sql)


In [ ]:
# получаем части адреса для каждого уникального full_address
cdi_raw_records = []
cdi_error_records = []
cdi_result_columns = None

with khd_connection.cursor() as cursor:
    for row_number, row in enumerate(unique_addresses.itertuples(index=False), start=1):
        try:
            cursor.execute(cdi_function_sql, address_text=row.source_address)
            result_columns = [str(column[0]).lower() for column in cursor.description]
            if cdi_result_columns is None:
                cdi_result_columns = result_columns

            while True:
                batch = cursor.fetchmany(100)
                if not batch:
                    break
                for values in batch:
                    record = dict(zip(result_columns, values))
                    record['address_lookup_id'] = int(row.address_lookup_id)
                    record['sphere_full_address'] = row.source_address
                    cdi_raw_records.append(record)
        except oracledb.Error as error:
            error_text = str(error)
            cdi_error_records.append({
                'address_lookup_id': int(row.address_lookup_id),
                'sphere_full_address': row.source_address,
                'error': error_text,
            })
            if any(code in error_text for code in ['ORA-00904', 'ORA-00942', 'ORA-06550', 'ORA-01031']):
                raise RuntimeError('Функция CDI недоступна. Ошибка Oracle: ' + error_text) from error

        if row_number % 100 == 0:
            print('Обработано адресов:', row_number, 'из', len(unique_addresses))

cdi_raw_df = pd.DataFrame(cdi_raw_records)
cdi_errors_df = pd.DataFrame(cdi_error_records)

if cdi_result_columns is None and not unique_addresses.empty:
    raise RuntimeError('CDI не вернул структуру результата')

print('Строк в ответах CDI:', len(cdi_raw_df))
print('Ошибок CDI:', len(cdi_errors_df))


In [ ]:
# находим нужные колонки в фактическом ответе CDI
def existing_columns(frame, names):
    return [name for name in names if name in frame.columns]


def first_filled(frame, names):
    columns = existing_columns(frame, names)
    if not columns:
        return pd.Series(pd.NA, index=frame.index, dtype='string')

    result = pd.Series(pd.NA, index=frame.index, dtype='string')
    for column in columns:
        values = frame[column].astype('string').str.strip().replace('', pd.NA)
        result = result.fillna(values)
    return result


if cdi_raw_df.empty:
    cdi_raw_df = pd.DataFrame(columns=['address_lookup_id', 'sphere_full_address'])

cdi_raw_df['cdi_region_raw'] = first_filled(
    cdi_raw_df, ['region', 'region_with_type']
)
cdi_raw_df['cdi_city_raw'] = first_filled(
    cdi_raw_df, ['city', 'city_with_type']
)
cdi_raw_df['cdi_settlement_raw'] = first_filled(
    cdi_raw_df, ['settlement', 'settlement_with_type']
)
cdi_raw_df['cdi_street_raw'] = first_filled(
    cdi_raw_df, ['street', 'street_with_type']
)
cdi_raw_df['cdi_house_raw'] = first_filled(
    cdi_raw_df, ['house', 'house_number']
)
cdi_raw_df['cdi_corpus_raw'] = first_filled(
    cdi_raw_df, ['corpus', 'house_corpus']
)
cdi_raw_df['cdi_structure_raw'] = first_filled(
    cdi_raw_df, ['building', 'structure', 'house_building']
)

# в ответе CDI block может означать корпус или строение
block_value = first_filled(cdi_raw_df, ['block'])
block_type = first_filled(cdi_raw_df, ['block_type_full', 'block_type']).str.lower()
is_structure = block_type.str.contains(r'стр|строен|соор', regex=True, na=False)
is_corpus = block_type.str.contains(r'корп|^к$', regex=True, na=False)
cdi_raw_df.loc[is_structure, 'cdi_structure_raw'] = (
    cdi_raw_df.loc[is_structure, 'cdi_structure_raw']
    .fillna(block_value.loc[is_structure])
)
cdi_raw_df.loc[is_corpus, 'cdi_corpus_raw'] = (
    cdi_raw_df.loc[is_corpus, 'cdi_corpus_raw']
    .fillna(block_value.loc[is_corpus])
)
cdi_raw_df['cdi_house_fias'] = first_filled(
    cdi_raw_df,
    ['house_fias_id', 'fias_id_house', 'fias_house_id'],
)

cdi_raw_df['cdi_premise_raw'] = first_filled(
    cdi_raw_df,
    ['flat', 'office', 'room', 'compartment', 'apartment'],
)

print('Колонки CDI:', ', '.join(cdi_result_columns or []))


In [ ]:
# приводим части адреса к одному виду
def normalize_component(value, component):
    if pd.isna(value):
        return pd.NA

    text_value = str(value).lower().replace('ё', 'е')
    text_value = re.sub(r'[^0-9a-zа-я]+', ' ', text_value)
    text_value = re.sub(r'\s+', ' ', text_value).strip()

    stop_words = {
        'region': {'область', 'обл', 'край', 'республика', 'респ', 'город', 'г'},
        'city': {
            'город', 'г', 'поселок', 'посёлок', 'п', 'рабочий',
            'рп', 'деревня', 'д', 'село', 'с', 'хутор', 'х',
            'станица', 'ст', 'территория', 'муниципальный', 'округ',
        },
        'settlement': {
            'город', 'г', 'поселок', 'посёлок', 'п', 'рабочий',
            'рп', 'деревня', 'д', 'село', 'с', 'хутор', 'х',
            'станица', 'ст', 'территория', 'муниципальный', 'округ',
        },
        'street': {
            'улица', 'ул', 'проспект', 'пр', 'проезд', 'переулок',
            'кт', 'д', 'пер', 'шоссе', 'ш', 'набережная', 'наб', 'бульвар',
            'бул', 'площадь', 'пл', 'микрорайон', 'мкр', 'аллея',
        },
        'house': {'дом', 'д', 'владение', 'вл'},
        'corpus': {'корпус', 'корп', 'к'},
        'structure': {'строение', 'стр', 'сооружение', 'соор'},
        'premise': {
            'квартира', 'кв', 'офис', 'помещение', 'пом',
            'комната', 'комн', 'апартамент', 'апартаменты',
        },
    }
    words = [word for word in text_value.split() if word not in stop_words.get(component, set())]
    normalized = ' '.join(words).strip()
    return normalized if normalized else pd.NA


component_names = [
    'region', 'city', 'settlement', 'street',
    'house', 'corpus', 'structure', 'premise',
]
for component in component_names:
    cdi_raw_df[f'cdi_{component}_norm'] = cdi_raw_df[f'cdi_{component}_raw'].map(
        lambda value, name=component: normalize_component(value, name)
    )

# одинаковые варианты разбора одного адреса считаем одним вариантом
parsed_columns = [f'cdi_{name}_norm' for name in component_names]
variant_key_columns = parsed_columns + ['cdi_house_fias']
cdi_parsed_variants = (
    cdi_raw_df[['address_lookup_id', 'sphere_full_address'] +
               [f'cdi_{name}_raw' for name in component_names]
               + parsed_columns + ['cdi_house_fias']]
    .drop_duplicates(['address_lookup_id'] + variant_key_columns)
)

cdi_variant_count = (
    cdi_parsed_variants.groupby('address_lookup_id')
    .size()
    .rename('cdi_parsed_variant_count')
    .reset_index()
)

cdi_unique = (
    cdi_parsed_variants
    .merge(cdi_variant_count, on='address_lookup_id', how='left')
    .loc[lambda frame: frame['cdi_parsed_variant_count'].eq(1)]
    .drop_duplicates('address_lookup_id')
)

cdi_lookup_df = (
    unique_addresses
    .rename(columns={'source_address': 'sphere_full_address'})
    .merge(cdi_variant_count, on='address_lookup_id', how='left')
    .merge(
        cdi_unique.drop(
            columns=['sphere_full_address', 'cdi_parsed_variant_count'],
            errors='ignore',
        ),
        on='address_lookup_id', how='left', validate='one_to_one',
    )
)
cdi_lookup_df['cdi_parsed_variant_count'] = (
    cdi_lookup_df['cdi_parsed_variant_count'].fillna(0).astype('int64')
)
cdi_lookup_df['cdi_parse_status'] = 'not_found'
cdi_lookup_df.loc[
    cdi_lookup_df['cdi_parsed_variant_count'].eq(1), 'cdi_parse_status'
] = 'parsed'
cdi_lookup_df.loc[
    cdi_lookup_df['cdi_parsed_variant_count'].gt(1), 'cdi_parse_status'
] = 'ambiguous_cdi'

if not cdi_errors_df.empty:
    error_ids = set(cdi_errors_df['address_lookup_id'])
    cdi_lookup_df.loc[
        cdi_lookup_df['address_lookup_id'].isin(error_ids), 'cdi_parse_status'
    ] = 'lookup_error'

display(cdi_lookup_df['cdi_parse_status'].value_counts(dropna=False))


In [ ]:
# возвращаем разобранный адрес к строкам датасета
cdi_address_df = sphere_with_row_id.merge(
    cdi_lookup_df,
    left_on='source_address',
    right_on='sphere_full_address',
    how='left',
    validate='many_to_one',
)
cdi_address_df['cdi_parse_status'] = (
    cdi_address_df['cdi_parse_status'].fillna('no_source_address')
)
cdi_address_df['sphere_area'] = pd.to_numeric(
    cdi_address_df['total_area']
    .astype('string')
    .str.replace(',', '.', regex=False),
    errors='coerce',
)

has_minimum_address = (
    cdi_address_df['cdi_house_norm'].notna()
    & (
        cdi_address_df['cdi_street_norm'].notna()
        | cdi_address_df['cdi_city_norm'].notna()
        | cdi_address_df['cdi_settlement_norm'].notna()
    )
)
has_house_fias = (
    cdi_address_df['cdi_house_fias']
    .astype('string')
    .str.strip()
    .notna()
)

cdi_address_df['pre_match_status'] = cdi_address_df['cdi_parse_status']
cdi_address_df.loc[
    cdi_address_df['cdi_parse_status'].eq('not_found'),
    'pre_match_status',
] = 'address_not_parsed'
cdi_address_df.loc[
    cdi_address_df['cdi_parse_status'].eq('parsed') & ~has_minimum_address,
    'pre_match_status',
] = 'address_not_parsed'
cdi_address_df.loc[
    cdi_address_df['cdi_parse_status'].eq('parsed')
    & has_minimum_address
    & ~has_house_fias,
    'pre_match_status',
] = 'cdi_fias_not_found'
cdi_address_df.loc[
    cdi_address_df['cdi_parse_status'].eq('parsed')
    & has_minimum_address
    & has_house_fias
    & (
        cdi_address_df['sphere_area'].isna()
        | cdi_address_df['sphere_area'].le(0)
    ),
    'pre_match_status',
] = 'no_area'
cdi_address_df.loc[
    cdi_address_df['cdi_parse_status'].eq('parsed')
    & has_minimum_address
    & has_house_fias
    & cdi_address_df['sphere_area'].gt(0),
    'pre_match_status',
] = 'ready_for_egrn'

print('Строк после CDI:', len(cdi_address_df))
display(cdi_address_df['pre_match_status'].value_counts(dropna=False))


# 6. Быстрый поиск в `EGRN_DATA`

Сначала ЕГРН отбирается по ФИАС дома. Это сильно сокращает количество проверяемых записей.

Затем кандидаты проверяются по частям адреса и площади. Если помещение указано в CDI, оно обязательно должно совпасть. Если помещение не указано, оно не участвует в проверке.


In [ ]:
# готовим строки для поиска в ЕГРН
match_input_df = cdi_address_df.loc[
    cdi_address_df['pre_match_status'].eq('ready_for_egrn'),
    ['sphere_row_id', 'sphere_area', 'cdi_house_fias'] + parsed_columns,
].copy()


def json_value(value):
    if pd.isna(value):
        return None
    if hasattr(value, 'item'):
        value = value.item()
    return value


match_records = []
for row in match_input_df.itertuples(index=False):
    match_records.append({
        'sphere_row_id': int(row.sphere_row_id),
        'sphere_area': float(row.sphere_area),
        'fias_id_house': str(row.cdi_house_fias).strip(),
        'region_norm': json_value(row.cdi_region_norm),
        'city_norm': json_value(row.cdi_city_norm),
        'settlement_norm': json_value(row.cdi_settlement_norm),
        'street_norm': json_value(row.cdi_street_norm),
        'house_norm': json_value(row.cdi_house_norm),
        'corpus_norm': json_value(row.cdi_corpus_norm),
        'structure_norm': json_value(row.cdi_structure_norm),
        'premise_norm': json_value(row.cdi_premise_norm),
    })

objects_json = json.dumps(match_records, ensure_ascii=False)

print('Строк для поиска в ЕГРН:', len(match_records))
display(match_input_df.head(5))


In [ ]:
# сначала отбираем кандидатов по ФИАС
# адрес и площадь проверяются только внутри этого небольшого списка
egrn_address_sql = r"""
with sphere_objects as (
    select /*+ materialize */
        s.sphere_row_id,
        s.sphere_area,
        trim(s.fias_id_house) as fias_id_house,
        s.region_norm,
        s.city_norm,
        s.settlement_norm,
        s.street_norm,
        s.house_norm,
        s.corpus_norm,
        s.structure_norm,
        s.premise_norm
    from json_table(
        :objects_json,
        '$[*]'
        columns (
            sphere_row_id number path '$.sphere_row_id',
            sphere_area number path '$.sphere_area',
            fias_id_house varchar2(500) path '$.fias_id_house',
            region_norm varchar2(500) path '$.region_norm',
            city_norm varchar2(500) path '$.city_norm',
            settlement_norm varchar2(500) path '$.settlement_norm',
            street_norm varchar2(500) path '$.street_norm',
            house_norm varchar2(200) path '$.house_norm',
            corpus_norm varchar2(200) path '$.corpus_norm',
            structure_norm varchar2(200) path '$.structure_norm',
            premise_norm varchar2(200) path '$.premise_norm'
        )
    ) s
),

fias_candidates as (
    select /*+ no_parallel(e) */
        s.sphere_row_id,
        s.sphere_area,
        s.fias_id_house as cdi_fias_id_house,
        s.region_norm as cdi_region_norm,
        s.city_norm as cdi_city_norm,
        s.settlement_norm as cdi_settlement_norm,
        s.street_norm as cdi_street_norm,
        s.house_norm as cdi_house_norm,
        s.corpus_norm as cdi_corpus_norm,
        s.structure_norm as cdi_structure_norm,
        s.premise_norm as cdi_premise_norm,
        coalesce(
            nullif(trim(e.cadaster), ''),
            'CAD_IND:' || cast(e.cad_ind as varchar2(200))
        ) as egrn_object_key,
        e.cad_ind,
        e.cadaster,
        e.egrn_address,
        e.square,
        case
            when regexp_like(
                replace(trim(cast(e.square as varchar2(200))), ',', '.'),
                '^[0-9]+([.][0-9]+)?$'
            )
            then to_number(
                replace(trim(cast(e.square as varchar2(200))), ',', '.'),
                '999999999999999999999999D9999999999',
                q'[NLS_NUMERIC_CHARACTERS='.,']'
            )
        end as egrn_area,
        e.region,
        e.city,
        e.settlement,
        e.street,
        e.house_number,
        e.vladenie,
        e.korpus,
        e.stroenie,
        e.flat,
        e.flat2,
        e.office,
        e.office2,
        e.room,
        e.room2,
        e.compartment1,
        e.compartment2,
        e.building_type,
        e.oks_type,
        e.oks_purpose,
        e.object_status,
        e.fias_level,
        e.fias_id_house as egrn_fias_id_house,
        e.row_update_date,
        e.ias_update_date,
        regexp_replace(regexp_replace(replace(lower(trim(e.region)), 'ё', 'е'), '[^0-9a-zа-я]+', ' '), '(^| )(область|обл|край|республика|респ|город|г)( |$)', ' ') as egrn_region_norm,
        regexp_replace(regexp_replace(replace(lower(trim(e.city)), 'ё', 'е'), '[^0-9a-zа-я]+', ' '), '(^| )(город|г|поселок|п|рабочий|рп|деревня|д|село|с|хутор|х|станица|ст|территория|муниципальный|округ)( |$)', ' ') as egrn_city_norm,
        regexp_replace(regexp_replace(replace(lower(trim(e.settlement)), 'ё', 'е'), '[^0-9a-zа-я]+', ' '), '(^| )(город|г|поселок|п|рабочий|рп|деревня|д|село|с|хутор|х|станица|ст|территория|муниципальный|округ)( |$)', ' ') as egrn_settlement_norm,
        regexp_replace(regexp_replace(replace(lower(trim(e.street)), 'ё', 'е'), '[^0-9a-zа-я]+', ' '), '(^| )(улица|ул|проспект|пр|кт|д|проезд|переулок|пер|шоссе|ш|набережная|наб|бульвар|бул|площадь|пл|микрорайон|мкр|аллея)( |$)', ' ') as egrn_street_norm,
        regexp_replace(regexp_replace(replace(lower(trim(e.house_number)), 'ё', 'е'), '[^0-9a-zа-я]+', ' '), '(^| )(дом|д|владение|вл)( |$)', ' ') as egrn_house_norm,
        regexp_replace(regexp_replace(replace(lower(trim(e.vladenie)), 'ё', 'е'), '[^0-9a-zа-я]+', ' '), '(^| )(дом|д|владение|вл)( |$)', ' ') as egrn_vladenie_norm,
        regexp_replace(regexp_replace(replace(lower(trim(e.korpus)), 'ё', 'е'), '[^0-9a-zа-я]+', ' '), '(^| )(корпус|корп|к)( |$)', ' ') as egrn_corpus_norm,
        regexp_replace(regexp_replace(replace(lower(trim(e.stroenie)), 'ё', 'е'), '[^0-9a-zа-я]+', ' '), '(^| )(строение|стр|сооружение|соор)( |$)', ' ') as egrn_structure_norm,
        regexp_replace(replace(lower(trim(e.flat)), 'ё', 'е'), '[^0-9a-zа-я]+', ' ') as egrn_flat_norm,
        regexp_replace(replace(lower(trim(e.flat2)), 'ё', 'е'), '[^0-9a-zа-я]+', ' ') as egrn_flat2_norm,
        regexp_replace(replace(lower(trim(e.office)), 'ё', 'е'), '[^0-9a-zа-я]+', ' ') as egrn_office_norm,
        regexp_replace(replace(lower(trim(e.office2)), 'ё', 'е'), '[^0-9a-zа-я]+', ' ') as egrn_office2_norm,
        regexp_replace(replace(lower(trim(e.room)), 'ё', 'е'), '[^0-9a-zа-я]+', ' ') as egrn_room_norm,
        regexp_replace(replace(lower(trim(e.room2)), 'ё', 'е'), '[^0-9a-zа-я]+', ' ') as egrn_room2_norm,
        regexp_replace(replace(lower(trim(e.compartment1)), 'ё', 'е'), '[^0-9a-zа-я]+', ' ') as egrn_compartment1_norm,
        regexp_replace(replace(lower(trim(e.compartment2)), 'ё', 'е'), '[^0-9a-zа-я]+', ' ') as egrn_compartment2_norm
    from sphere_objects s
    join DM_RISK_AVATAR.EGRN_DATA e
        on e.fias_id_house = s.fias_id_house
    where e.cadaster is not null or e.cad_ind is not null
),

ranked_versions as (
    select
        candidate.*,
        row_number() over (
            partition by candidate.sphere_row_id, candidate.egrn_object_key
            order by
                candidate.row_update_date desc nulls last,
                candidate.ias_update_date desc nulls last,
                candidate.cad_ind desc nulls last
        ) as version_number
    from fias_candidates candidate
),

current_candidates as (
    select candidate.*
    from ranked_versions candidate
    where candidate.version_number = 1
),

address_area_matches as (
    select candidate.*
    from current_candidates candidate
    where (
            candidate.cdi_region_norm is null
            or trim(candidate.egrn_region_norm) = candidate.cdi_region_norm
          )
      and (
            candidate.cdi_city_norm is null
            or trim(candidate.egrn_city_norm) = candidate.cdi_city_norm
            or (
                candidate.egrn_city_norm is null
                and trim(candidate.egrn_settlement_norm) = candidate.cdi_city_norm
            )
          )
      and (
            candidate.cdi_settlement_norm is null
            or trim(candidate.egrn_settlement_norm) = candidate.cdi_settlement_norm
            or (
                candidate.egrn_settlement_norm is null
                and trim(candidate.egrn_city_norm) = candidate.cdi_settlement_norm
            )
          )
      and (
            candidate.cdi_street_norm is null
            or trim(candidate.egrn_street_norm) = candidate.cdi_street_norm
          )
      and (
            trim(candidate.egrn_house_norm) = candidate.cdi_house_norm
            or trim(candidate.egrn_vladenie_norm) = candidate.cdi_house_norm
          )
      and (
            candidate.cdi_corpus_norm is null
            or trim(candidate.egrn_corpus_norm) = candidate.cdi_corpus_norm
          )
      and (
            candidate.cdi_structure_norm is null
            or trim(candidate.egrn_structure_norm) = candidate.cdi_structure_norm
          )
      and (
            candidate.cdi_premise_norm is null
            or trim(candidate.egrn_flat_norm) = candidate.cdi_premise_norm
            or trim(candidate.egrn_flat2_norm) = candidate.cdi_premise_norm
            or trim(candidate.egrn_office_norm) = candidate.cdi_premise_norm
            or trim(candidate.egrn_office2_norm) = candidate.cdi_premise_norm
            or trim(candidate.egrn_room_norm) = candidate.cdi_premise_norm
            or trim(candidate.egrn_room2_norm) = candidate.cdi_premise_norm
            or trim(candidate.egrn_compartment1_norm) = candidate.cdi_premise_norm
            or trim(candidate.egrn_compartment2_norm) = candidate.cdi_premise_norm
          )
      and candidate.egrn_area is not null
      and abs(candidate.egrn_area - candidate.sphere_area)
          <= greatest(1, candidate.sphere_area * 0.01)
)

select /*+ no_parallel */
    candidate.*
from address_area_matches candidate
order by candidate.sphere_row_id, candidate.egrn_object_key
"""


In [ ]:
# выполняем один запрос вместо многократного просмотра EGRN_DATA
khd_schema = KHD_DATA_SCHEMA.upper()
if not khd_schema.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

egrn_query = egrn_address_sql.replace(
    'DM_RISK_AVATAR.', f'{khd_schema}.'
)

if match_records:
    try:
        with khd_connection.cursor() as cursor:
            cursor.execute('alter session disable parallel query')
            bind = cursor.var(oracledb.DB_TYPE_CLOB)
            bind.setvalue(0, objects_json)
            cursor.execute(egrn_query, objects_json=bind)
            columns = [str(column[0]).lower() for column in cursor.description]
            rows = cursor.fetchall()
        egrn_candidates_df = pd.DataFrame(rows, columns=columns)
    except oracledb.Error as error:
        raise RuntimeError('Ошибка поиска кандидатов ЕГРН: ' + str(error)) from error
else:
    egrn_candidates_df = pd.DataFrame(
        columns=['sphere_row_id', 'egrn_object_key']
    )

required_candidate_columns = {'sphere_row_id', 'egrn_object_key'}
missing_candidate_columns = sorted(
    required_candidate_columns - set(egrn_candidates_df.columns)
)
if missing_candidate_columns:
    raise ValueError(
        'ЕГРН не вернул колонки: ' + ', '.join(missing_candidate_columns)
    )
if egrn_candidates_df.duplicated(
    ['sphere_row_id', 'egrn_object_key']
).any():
    raise ValueError(
        'После выбора свежей версии остались повторы объекта ЕГРН'
    )

print('Найдено кандидатов после проверки адреса и площади:', len(egrn_candidates_df))


# 7. Выбор единичной связи

Один кандидат — данные ЕГРН присоединяются. Ноль или несколько кандидатов — данные ЕГРН остаются пустыми.


In [ ]:
# считаем кадастровые объекты для каждой строки Сферы
if egrn_candidates_df.empty:
    candidate_count_df = pd.DataFrame(columns=['sphere_row_id', 'egrn_candidate_count'])
    unique_egrn_df = egrn_candidates_df.copy()
else:
    candidate_count_df = (
        egrn_candidates_df.groupby('sphere_row_id')['egrn_object_key']
        .nunique()
        .rename('egrn_candidate_count')
        .reset_index()
    )
    unique_ids = set(
        candidate_count_df.loc[
            candidate_count_df['egrn_candidate_count'].eq(1),
            'sphere_row_id',
        ]
    )
    unique_egrn_df = egrn_candidates_df.loc[
        egrn_candidates_df['sphere_row_id'].isin(unique_ids)
    ].drop_duplicates('sphere_row_id')

result_df = cdi_address_df.merge(
    candidate_count_df,
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)
result_df['egrn_candidate_count'] = (
    result_df['egrn_candidate_count'].fillna(0).astype('int64')
)

egrn_columns = [
    column for column in unique_egrn_df.columns
    if column != 'sphere_row_id' and column not in cdi_address_df.columns
]
result_df = result_df.merge(
    unique_egrn_df[['sphere_row_id'] + egrn_columns],
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)

result_df['connection_is_unique'] = result_df['egrn_candidate_count'].eq(1).astype('int64')
result_df['connection_method'] = result_df['pre_match_status']
result_df.loc[
    result_df['pre_match_status'].eq('ready_for_egrn')
    & result_df['egrn_candidate_count'].eq(0),
    'connection_method',
] = 'not_found'
result_df.loc[
    result_df['pre_match_status'].eq('ready_for_egrn')
    & result_df['egrn_candidate_count'].gt(1),
    'connection_method',
] = 'ambiguous'
result_df.loc[
    result_df['pre_match_status'].eq('ready_for_egrn')
    & result_df['egrn_candidate_count'].eq(1),
    'connection_method',
] = 'address_parts_and_area'

if len(result_df) != len(expanded_df):
    raise ValueError('После соединения изменилось количество строк')
if result_df['sphere_row_id'].duplicated().any():
    raise ValueError('После соединения появились повторные строки')

print('Строк в результате:', len(result_df))
display(result_df['connection_method'].value_counts(dropna=False))


# 8. Проверка результата

В таблице ниже рядом показаны части адреса CDI и ЕГРН. Так можно увидеть, по каким значениям выполнено сравнение.


In [ ]:
check_columns = [
    'sphere_row_id', 'contract_id', 'contract_number', 'object_id',
    'characteristics_id', 'geo_address_id', 'source_address',
    'cdi_house_fias', 'egrn_fias_id_house',
    'cdi_region_raw', 'region',
    'cdi_city_raw', 'city',
    'cdi_settlement_raw', 'settlement',
    'cdi_street_raw', 'street',
    'cdi_house_raw', 'house_number', 'vladenie',
    'cdi_corpus_raw', 'korpus',
    'cdi_structure_raw', 'stroenie',
    'cdi_premise_raw', 'flat', 'office', 'room',
    'sphere_area', 'egrn_area', 'square',
    'egrn_candidate_count', 'connection_is_unique', 'connection_method',
    'cad_ind', 'cadaster', 'egrn_address', 'oks_type', 'oks_purpose',
]
check_columns = [column for column in check_columns if column in result_df.columns]
connection_check_df = result_df[check_columns].copy()

quality_profile = pd.DataFrame({
    'Показатель': [
        'Строк в датасете',
        'Единичных связей с ЕГРН',
        'Не найдено в ЕГРН',
        'Найдено несколько объектов',
        'Нет площади',
        'Нет исходного адреса',
        'CDI не разобрал адрес',
        'CDI не вернул ФИАС дома',
    ],
    'Значение': [
        len(result_df),
        result_df['connection_is_unique'].sum(),
        result_df['connection_method'].eq('not_found').sum(),
        result_df['connection_method'].eq('ambiguous').sum(),
        result_df['connection_method'].eq('no_area').sum(),
        result_df['connection_method'].eq('no_source_address').sum(),
        result_df['connection_method'].isin(['address_not_parsed', 'ambiguous_cdi', 'lookup_error']).sum(),
        result_df['connection_method'].eq('cdi_fias_not_found').sum(),
    ],
})

display(quality_profile)
display(connection_check_df.head(20))


# 9. Сохранение файлов

Основной файл сохраняет все строки датасета. Проверочный файл содержит только поля, нужные для ручной проверки соединения.


In [ ]:
snapshot_at = pd.Timestamp.now(tz='Europe/Moscow').isoformat()
result_df['external_snapshot_at'] = snapshot_at

final_path = OUTPUT_DIR / 'датасет_CDI_ЕГРН_адрес_и_площадь.csv'
check_path = OUTPUT_DIR / 'проверка_CDI_ЕГРН_адрес_и_площадь.csv'
candidates_path = OUTPUT_DIR / 'кандидаты_CDI_ЕГРН_адрес_и_площадь.csv'
cdi_path = OUTPUT_DIR / 'разбор_адресов_CDI.csv'
cdi_errors_path = OUTPUT_DIR / 'ошибки_CDI.csv'

result_df.to_csv(final_path, index=False, sep=';', encoding='utf-8-sig')
connection_check_df.to_csv(check_path, index=False, sep=';', encoding='utf-8-sig')
egrn_candidates_df.to_csv(candidates_path, index=False, sep=';', encoding='utf-8-sig')
cdi_lookup_df.to_csv(cdi_path, index=False, sep=';', encoding='utf-8-sig')
cdi_errors_df.to_csv(cdi_errors_path, index=False, sep=';', encoding='utf-8-sig')

print('Итоговый датасет:', final_path)
print('Проверка соединения:', check_path)
print('Все кандидаты ЕГРН:', candidates_path)
print('Разбор адресов CDI:', cdi_path)
print('Ошибки CDI:', cdi_errors_path)


# 10. Список уникальных ИНН

Пустые значения и повторы исключаются.


In [ ]:
inn_column = next(
    (column for column in ['policyholder_inn', 'inn'] if column in result_df.columns),
    None,
)
if inn_column is None:
    raise ValueError('В итоговом датасете не найдена колонка ИНН')

inn_df = (
    result_df[[inn_column]]
    .rename(columns={inn_column: 'inn'})
    .assign(inn=lambda frame: frame['inn'].astype('string').str.strip())
    .loc[lambda frame: frame['inn'].notna() & frame['inn'].ne('')]
    .drop_duplicates()
    .sort_values('inn')
    .reset_index(drop=True)
)

inn_path = OUTPUT_DIR / 'inn.csv'
inn_df.to_csv(inn_path, index=False, sep=';', encoding='utf-8-sig')

print('Уникальных ИНН:', len(inn_df))
print('Файл:', inn_path)


In [ ]:
engine.dispose()
khd_connection.close()
print('Подключения закрыты')
